<a href="https://colab.research.google.com/github/jessicalewinter/deep-learning-specialization/blob/main/course-notebooks/machine-learning/quiz4_implemtation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Model Training and Evaluation

In [2]:
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, input_size=10, hidden_size=64, output_size=1):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Initialize model, loss, and Adam optimizer
model = MLP()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) # Added weight_decay

# 2. Configure the learning rate scheduler
# Reduces LR by half if validation loss plateaus for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

# --- Dummy DataLoaders for demonstration --- START
# You should replace these with your actual training and validation datasets and DataLoaders
input_size = 10 # Must match the input_size of your MLP model
num_train_samples = 500
num_val_samples = 100

dummy_train_data = torch.randn(num_train_samples, input_size)
dummy_train_labels = torch.randint(0, 2, (num_train_samples, 1)).float() # Binary labels
train_dataset = TensorDataset(dummy_train_data, dummy_train_labels)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

dummy_val_data = torch.randn(num_val_samples, input_size)
dummy_val_labels = torch.randint(0, 2, (num_val_samples, 1)).float() # Binary labels
val_dataset = TensorDataset(dummy_val_data, dummy_val_labels)
val_loader = DataLoader(val_dataset, batch_size=32)
# --- Dummy DataLoaders --- END

# 3. Training Loop
for epoch in range(50):
    model.train() # Set model to training mode
    train_loss = 0.0
    for x_train, y_train in train_loader:
        optimizer.zero_grad()
        outputs = model(x_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # 4. Validation phase
    model.eval() # Set model to evaluation mode
    val_loss = 0.0
    with torch.no_grad():
        for x_val, y_val in val_loader:
            outputs = model(x_val)
            loss = criterion(outputs, y_val)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    # 5. Adapt learning rate based on validation loss
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

Epoch 1 | Train Loss: 0.6977 | Val Loss: 0.7045
Epoch 2 | Train Loss: 0.6917 | Val Loss: 0.7006
Epoch 3 | Train Loss: 0.6885 | Val Loss: 0.6973
Epoch 4 | Train Loss: 0.6858 | Val Loss: 0.6999
Epoch 5 | Train Loss: 0.6827 | Val Loss: 0.7010
Epoch 6 | Train Loss: 0.6786 | Val Loss: 0.7010
Epoch 7 | Train Loss: 0.6757 | Val Loss: 0.7032
Epoch 8 | Train Loss: 0.6719 | Val Loss: 0.7086
Epoch 9 | Train Loss: 0.6691 | Val Loss: 0.7036
Epoch 10 | Train Loss: 0.6662 | Val Loss: 0.7092
Epoch 11 | Train Loss: 0.6646 | Val Loss: 0.7095
Epoch 12 | Train Loss: 0.6612 | Val Loss: 0.7107
Epoch 13 | Train Loss: 0.6604 | Val Loss: 0.7073
Epoch 14 | Train Loss: 0.6588 | Val Loss: 0.7105
Epoch 15 | Train Loss: 0.6583 | Val Loss: 0.7141
Epoch 16 | Train Loss: 0.6561 | Val Loss: 0.7133
Epoch 17 | Train Loss: 0.6555 | Val Loss: 0.7119
Epoch 18 | Train Loss: 0.6544 | Val Loss: 0.7118
Epoch 19 | Train Loss: 0.6549 | Val Loss: 0.7116
Epoch 20 | Train Loss: 0.6537 | Val Loss: 0.7124
Epoch 21 | Train Loss: 0.6526

# Perceptron

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

# =====================================================================
# 1. READ DATA FROM CSV (Just like Iris dataset workflows)
# =====================================================================
# Load the CSV file into a Pandas DataFrame
df = pd.read_csv('risk_data.csv')

# Extract features (x1: income, x2: debt_percentage)
# We select all rows, and the first two columns
X_raw = df[['income', 'debt_percentage']].values

# Extract target labels (risk_label)
y_raw = df[['risk_label']].values

# Convert the NumPy arrays from Pandas into PyTorch Tensors
X = torch.tensor(X_raw, dtype=torch.float32)
y = torch.tensor(y_raw, dtype=torch.float32)

# =====================================================================
# 2. DEFINE THE PERCEPTRON MODEL
# =====================================================================
class Perceptron(nn.Module):
    def __init__(self):
        super(Perceptron, self).__init__()
        self.linear = nn.Linear(in_features=2, out_features=1)

    def forward(self, x):
        return self.linear(x)

# Initialize Model, Loss Function, and Optimizer
model = Perceptron()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# =====================================================================
# 3. TRAINING LOOP
# =====================================================================
for epoch in range(500):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

print("Training completed using data imported from CSV!")

I've added a basic `YourMLPModel` class, which is a simple Multi-Layer Perceptron suitable for binary classification with `BCELoss`. You'll need to ensure your `input_size` in the model's constructor matches the feature dimension of your input data. Also, you'll need to define `val_loader` with your validation dataset.